# 1. Widget definitions

In [0]:
dbutils.widgets.dropdown("env", "staging", ["dev", "staging", "prod"], "Environment")
dbutils.widgets.text("domain", "salesforce", "Domain Name")
dbutils.widgets.text("client_id", "abcd-defg-hijlk-mnop", "Databricks Client Id")

env = dbutils.widgets.get("env")
domain = dbutils.widgets.get("domain")
client_id = dbutils.widgets.get("client_id")

# 2. Target naming convention

In [0]:
catalog_name = f"{domain}_{env}"
dbt_schema_name = f"dbt_{env}"
gold_schema_name = "gold"

In [0]:
query = f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{dbt_schema_name}"
spark.sql(query)

# 3. Required Unity Catalog grants

#### Step A: Grant catalog traversal

In [0]:
query = f"GRANT USE CATALOG ON CATALOG {catalog_name} TO `{client_id}`"
spark.sql(query)

#### Step B: Grant schema traversal on gold

In [0]:
spark.sql(f"GRANT USE SCHEMA, SELECT ON SCHEMA {catalog_name}.{gold_schema_name} TO `{client_id}`")

#### Step C: Grant use schema, create table select and modify on schema inside the dbt schema

In [0]:
spark.sql(f"GRANT USE SCHEMA, CREATE TABLE, SELECT, MODIFY ON SCHEMA {catalog_name}.{dbt_schema_name} TO `{client_id}`")

### VERIFY

In [0]:
display(spark.sql(f"SHOW TABLES IN {catalog_name}.gold"))

In [0]:
query = f"""
SELECT table_name, column_name, data_type
FROM {catalog_name}.information_schema.columns
WHERE table_schema = 'gold'
ORDER BY table_name, ordinal_position;
"""
spark.sql(query)

In [0]:
query = f"SHOW GRANTS ON CATALOG {catalog_name}"
spark.sql(query)

In [0]:
query = f"SHOW GRANTS `{client_id}` ON SCHEMA {catalog_name}.{dbt_schema_name}"
spark.sql(query)

In [0]:
query = f"SHOW GRANTS `{client_id}` ON SCHEMA {catalog_name}.{gold_schema_name}"
spark.sql(query)